### BASELINE - ALGORITMO GENÉTICO

#### IMPORTS

In [17]:
import random
import math
import time
from dataclasses import dataclass, field
from typing import List, Tuple

from baseline.POSSIBLE import DISTRICTS_POINTS as DP

### Parâmetros do problema

In [18]:
NUMBER_AMBUS_TYPE_A = 1
NUMBER_AMBUS_TYPE_B = 4
RAIO = 0.0084
#10min 15min, 30min, 1h
#0.0056 0.0084, 0.0168, 0.0336
BONUS_TYPE_A = 1.20

### Parâmetros do GA

In [19]:
POPULATION_SIZE = 200 
MAX_GENERATIONS = 500
PCT_ELITISM = 0.30 # preserva 30% dos melhores
PCT_RANDOM = 0.20 # 20% aleatórios para diversidade

DISASTER_EVERY = 50 # a cada 50 geracoes sem melhora, gera uma catastrofe
PCT_DISASTER_DEATH = 0.40 # % da população que morre na catastrofe

### Estruturas dos Dados

In [20]:
NUMBER_LOCATIONS = len(DP)
_keys = list(DP.keys())

COORDS_X = [DP[k][0] for k in _keys]
COORDS_Y = [DP[k][1] for k in _keys]
WEIGHTS = [float(DP[k][2]) if str(DP[k][2]) != "nan" else 0.0 for k in _keys]
TOTAL_DEMAND = sum(WEIGHTS)

@dataclass
class Solution:
    xA: List[int] = field(default_factory = lambda: [0] * NUMBER_LOCATIONS)
    xB: List[int] = field(default_factory = lambda: [0] * NUMBER_LOCATIONS)
    non_zeros_A: List[int] = field(default_factory=list)
    non_zeros_B: List[int] = field(default_factory=list)
    fitness: float = 0.0


    def __lt__(self, other: Solution): return self.fitness < other.fitness
    def __gt__(self, other: Solution): return self.fitness > other.fitness
    def __le__(self, other: Solution): return self.fitness <= other.fitness
    def __ge__(self, other: Solution): return self.fitness >= other.fitness


### Pré-computação da matriz de distâncias

In [21]:
def build_distance_matrix() -> List[List[float]]:
    mat = []
    for i in range(NUMBER_LOCATIONS):
        row = []
        for j in range(NUMBER_LOCATIONS):
            dx = COORDS_X[i] - COORDS_X[j]
            dy = COORDS_Y[i] - COORDS_Y[j]
            row.append(math.sqrt(dx*dx + dy*dy))
        mat.append(row)
    return mat

### Criação de soluções

In [22]:
def create_solution() -> Solution:
    """"
    Garante que as ambulâncias dos tipos A e B não 
    fiquem sobrepostas.

    Uma localização só pode sediar uma ambulância 
    """
    
    sol = Solution()
    pool = list(range(NUMBER_LOCATIONS))

    chosen_A = random.sample(pool, NUMBER_AMBUS_TYPE_A)
    for idx in chosen_A:
        sol.xA[idx] = 1
        sol.non_zeros_A.append(idx)

    remaining = [i for i in pool if i not in chosen_A]

    chosen_B = random.sample(remaining, NUMBER_AMBUS_TYPE_B)
    for idx in chosen_B:
        sol.xB[idx] = 1
        sol.non_zeros_B.append(idx)
    
    return sol

### Avaliação

In [23]:
def evaluate_solution(sol: Solution, dist_matrix: List[List[float]]) -> None:
    covered_A = set()
    covered_B = set()

    for a in sol.non_zeros_A:
        for j in range(NUMBER_LOCATIONS):
            if dist_matrix[a][j] <= RAIO:
                covered_A.add(j)

    for b in sol.non_zeros_B:
        for j in range(NUMBER_LOCATIONS):
            if dist_matrix[b][j] <= RAIO:
                covered_B.add(j)

    fitness = 0.0
    for j in covered_A:
        fitness += WEIGHTS[j] * BONUS_TYPE_A
    for j in covered_B - covered_A:
        fitness += WEIGHTS[j]

    sol.fitness = fitness

def coverage_percentage(sol: Solution, dist_matrix: List[List[float]]) -> float:
    """
    Retorna a % real da demanda coberta (sem bônus para ambulância do tipo A)
    """

    covered = set()
    for a in sol.non_zeros_A:
        for j in range(NUMBER_LOCATIONS):
            if dist_matrix[a][j] <= RAIO:
                covered.add(j)
    
    for b in sol.non_zeros_B:
        for j in range(NUMBER_LOCATIONS):
            if dist_matrix[b][j] <= RAIO:
                covered.add(j)
    
    return sum(WEIGHTS[j] for j in covered) / TOTAL_DEMAND * 100.0


def repair(sol: Solution) -> None:
    """"
    Garante que a solução tenha exatamente
     o número correto de ambulâncias de cada 
     tipo após crossover/mutação, sem sobreposição 
     entre tipos.

     Ordem de operações:
     1. Reconstruir non_zeros a partir dos vetores xA/xB
     2. Resolver sobreposições (mesma posição em A e B): remove de B,
     pois TypeA tem prioridade por ter maior bônus
     3. Ajustar excesso: remove aleatoriedade
     4. Ajustar falta: adiciona em posições livres (não usadas por nenhum tipo)
    """

    # Reconstruir listas a partir dos vetores binários
    sol.non_zeros_A = [i for i in range(NUMBER_LOCATIONS) if sol.xA[i] == 1]
    sol.non_zeros_B = [i for i in range(NUMBER_LOCATIONS) if sol.xB[i] == 1]

    # Resolver sobreposições
    overlap = set(sol.non_zeros_A) & set(sol.non_zeros_B)
    for idx in overlap:
        # TypeA tem prioridade - remove de B
        sol.xB[idx] = 0
        sol.non_zeros_B.remove(idx)

    # Ajustar TypeA
    while len(sol.non_zeros_A) > NUMBER_AMBUS_TYPE_A:
        remove = random.choice(sol.non_zeros_A)
        sol.xA[remove] = 0
        sol.non_zeros_A.remove(remove)

    occupied = set(sol.non_zeros_A) | set(sol.non_zeros_B)
    while len(sol.non_zeros_A) < NUMBER_AMBUS_TYPE_A:
        idx = random.randrange(NUMBER_LOCATIONS)
        if idx not in occupied:
            sol.xA[idx] = 1
            sol.non_zeros_A.append(idx)
            occupied.add(idx)

    # Ajustar TypeB
    while len(sol.non_zeros_B) > NUMBER_AMBUS_TYPE_B:
        remove = random.choice(sol.non_zeros_B)
        sol.xB[remove] = 0
        sol.non_zeros_B.remove(remove)

    occupied = set(sol.non_zeros_A) | set(sol.non_zeros_B)
    while len(sol.non_zeros_B) < NUMBER_AMBUS_TYPE_B:
        idx = random.randrange(NUMBER_LOCATIONS)
        if idx not in occupied:
            sol.xB[idx] = 1
            sol.non_zeros_B.append(idx)
            occupied.add(idx)

### Seleção de pais 

In [24]:
def roulette_select(population: List[Solution]) -> Solution:
    total = sum(s.fitness for s in population)
    if total == 0:
        return random.choice(population)
    
    pick = random.uniform(0, total)
    cumulative = 0.0
    for s in population:
        cumulative += s.fitness
        if cumulative >= pick:
            return s
    
    return population[-1]


### Crossover

In [25]:
def crossover(base: Solution, guide: Solution, dist_matrix: List[List[float]]) -> Solution:
    child = Solution()

    # TypeA
    for i in range(NUMBER_LOCATIONS):
        if base.xA[i] == guide.xA[i]:
            child.xA[i] = base.xA[i]
        elif base.xA[i] == 1:
            child.xA[i] = 1
        else:
            child.xA[i] = random.randint(0,1)

    # TypeB
    for i in range(NUMBER_LOCATIONS):
        if base.xB[i] == guide.xB[i]:
            child.xB[i] = base.xB[i]
        elif base.xB[i] == 1:
            child.xB[i] = 1
        else:
            child.xB[i] = random.randint(0,1)

    repair(child)
    evaluate_solution(child, dist_matrix)
    return child


### Mutação

In [26]:
def mutate(sol: Solution, dist_matrix: List[List[float]], mutation_rate: float = 0.05) -> None:
    """
    Troca aleatoriamente a posição de uma ambulância com prob. mutation_rate.
    A nova posição não pode estar ocupada por nenhum tipo (em sobreposição).
    """

    occupied = set(sol.non_zeros_A) | set(sol.non_zeros_B)

    if random.random() < mutation_rate and sol.non_zeros_A:
        old_pos = random.choice(sol.non_zeros_A)
        new_pos = random.randrange(NUMBER_LOCATIONS)
        if new_pos not in occupied or new_pos == old_pos:
            sol.xA[old_pos] = 0
            sol.non_zeros_A.remove(old_pos)
            occupied.discard(old_pos)
            sol.xA[new_pos] = 1
            sol.non_zeros_A.append(new_pos)
            occupied.add(new_pos)

    if random.random() < mutation_rate and sol.non_zeros_B:
        old_pos = random.choice(sol.non_zeros_B)
        new_pos = random.randrange(NUMBER_LOCATIONS)
        if new_pos not in occupied or new_pos == old_pos:
            sol.xB[old_pos] = 0
            sol.non_zeros_B.remove(old_pos)
            occupied.discard(old_pos)
            sol.xB[new_pos] = 1
            sol.non_zeros_B.append(new_pos)
            occupied.add(new_pos)
 
    evaluate_solution(sol, dist_matrix)


    

### Catástrofe

In [27]:
def disaster(population: List[Solution], dist_matrix: List[List[float]]) -> List[Solution]:
    n_die = int(PCT_DISASTER_DEATH * len(population))
    # mata os piores
    survivors = population[n_die:]
    for _ in range(n_die):
        s = create_solution()
        evaluate_solution(s, dist_matrix)
        survivors.append(s)
    return survivors

### FUNÇÃO PRINCIPAL

In [28]:
def run(seed: int = None, verbose: bool = False) -> Tuple[Solution, List[float]]:
    """
    Executa o GA e retorna (melhor_solução, histórico_de_fitness_por_geração).
    """
    if seed is not None:
        random.seed(seed)
 
    dist_matrix = build_distance_matrix()
 
    # População inicial
    population = []
    for _ in range(POPULATION_SIZE):
        s = create_solution()
        evaluate_solution(s, dist_matrix)
        population.append(s)
    population.sort()
 
    best = max(population, key=lambda s: s.fitness)
    history = [coverage_percentage(best, dist_matrix)]
    gens_without_improvement = 0
 
    for gen in range(MAX_GENERATIONS):
        new_pop = []
 
        # Elitismo
        n_elite = int(PCT_ELITISM * POPULATION_SIZE)
        new_pop.extend(population[-n_elite:])
 
        # Aleatórios para diversidade
        n_random = int(PCT_RANDOM * POPULATION_SIZE)
        for _ in range(n_random):
            s = create_solution()
            evaluate_solution(s, dist_matrix)
            new_pop.append(s)
 
        # Filhos por crossover
        n_children = POPULATION_SIZE - n_elite - n_random
        for _ in range(n_children):
            p1 = roulette_select(population)
            p2 = roulette_select(population)
            base  = p1 if p1.fitness >= p2.fitness else p2
            guide = p2 if base is p1 else p1
            child = crossover(base, guide, dist_matrix)
            mutate(child, dist_matrix)
            new_pop.append(child)
 
        new_pop.sort()
        population = new_pop
 
        gen_best = population[-1]
        if gen_best.fitness > best.fitness:
            best = gen_best
            gens_without_improvement = 0
        else:
            gens_without_improvement += 1
 
        cov = coverage_percentage(best, dist_matrix)
        history.append(cov)
 
        if verbose and gen % 50 == 0:
            print(f"  GA Gen {gen:4d} | Cobertura: {cov:.2f}%")
 
        # Catástrofe se parado
        if gens_without_improvement >= DISASTER_EVERY:
            population = disaster(population, dist_matrix)
            population.sort()
            gens_without_improvement = 0
 
    return best, history
 
 
if __name__ == "__main__":
    print("=== Algoritmo Genético (AG) ===")
    t0 = time.time()
    best_sol, hist = run(seed=42, verbose=True)
    elapsed = time.time() - t0
 
    dist_matrix = build_distance_matrix()
    cov = coverage_percentage(best_sol, dist_matrix)
    print(f"\nMelhor cobertura: {cov:.2f}%")
    print(f"Tempo: {elapsed:.1f}s")
    print(f"Posições TypeA: {best_sol.non_zeros_A}")
    print(f"Posições TypeB: {best_sol.non_zeros_B}")

=== Algoritmo Genético (AG) ===
  GA Gen    0 | Cobertura: 30.97%
  GA Gen   50 | Cobertura: 39.10%
  GA Gen  100 | Cobertura: 39.90%
  GA Gen  150 | Cobertura: 40.16%
  GA Gen  200 | Cobertura: 40.37%
  GA Gen  250 | Cobertura: 40.37%
  GA Gen  300 | Cobertura: 40.37%
  GA Gen  350 | Cobertura: 40.37%
  GA Gen  400 | Cobertura: 40.45%
  GA Gen  450 | Cobertura: 40.45%

Melhor cobertura: 40.59%
Tempo: 67.5s
Posições TypeA: [1907]
Posições TypeB: [467, 1235, 1442, 2178]
